# Transformer — BERT

In [ ]:
# Install Hugging Face Transformers

!pip install transformers datasets accelerate

In [ ]:
# Import BERT Tokenizer

from transformers import BertTokenizer

In [ ]:
# Load Pretrained BERT Tokenizer

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print("BERT tokenizer loaded successfully!")

In [ ]:
# Tokenize Sample Review

sample_text = df['cleaned_text'].iloc[0]

tokens = bert_tokenizer.tokenize(sample_text)

print(tokens[:30])

In [ ]:
# Encode Reviews for BERT

bert_inputs = bert_tokenizer(
    df['cleaned_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=100,
    return_tensors='pt'
)

print("BERT encoding completed successfully!")

In [ ]:
# Inspect Encoded Inputs

print("Input IDs Shape:", bert_inputs['input_ids'].shape)

print("Attention Mask Shape:", bert_inputs['attention_mask'].shape)

In [ ]:
# Import BERT Model

from transformers import BertForSequenceClassification

In [ ]:
# Load Pretrained BERT Classifier

bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

print("BERT model loaded successfully!")

In [ ]:
# Display BERT Architecture

print(bert_model)

In [ ]:
# Import PyTorch

import torch
from torch.utils.data import TensorDataset, DataLoader, random_split

In [ ]:
# Prepare Input Tensors

input_ids = bert_inputs['input_ids']
attention_masks = bert_inputs['attention_mask']

labels = torch.tensor(y)

print("Input IDs Shape:", input_ids.shape)
print("Attention Masks Shape:", attention_masks.shape)
print("Labels Shape:", labels.shape)

In [ ]:
# Create Tensor Dataset

dataset = TensorDataset(
    input_ids,
    attention_masks,
    labels
)

print("Dataset created successfully!")

In [ ]:
# Split Dataset for BERT

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

print("Training Size:", len(train_dataset))
print("Testing Size:", len(test_dataset))

In [ ]:
# Create DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16
)

print("DataLoaders created successfully!")

In [ ]:
# Move Model to Device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_model.to(device)

print("Using device:", device)

In [ ]:
# Configure Optimizer

from torch.optim import AdamW

optimizer = AdamW(
    bert_model.parameters(),
    lr=2e-5
)

print("Optimizer configured successfully!")

In [ ]:
# BERT Training Loop

from tqdm import tqdm

bert_model.train()

for epoch in range(2):

    total_loss = 0

    print(f"\nEpoch {epoch+1}")

    progress_bar = tqdm(train_loader)

    for batch in progress_bar:

        batch_input_ids = batch[0].to(device)
        batch_attention_masks = batch[1].to(device)
        batch_labels = batch[2].to(device)

        optimizer.zero_grad()

        outputs = bert_model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_masks,
            labels=batch_labels
        )

        loss = outputs.loss

        total_loss += loss.item()

        loss.backward()

        optimizer.step()

        progress_bar.set_postfix({'Loss': loss.item()})

    avg_loss = total_loss / len(train_loader)

    print("Average Training Loss:", avg_loss)

In [ ]:
# Evaluate BERT Model

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

bert_model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in test_loader:

        batch_input_ids = batch[0].to(device)
        batch_attention_masks = batch[1].to(device)
        batch_labels = batch[2].to(device)

        outputs = bert_model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_masks
        )

        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(batch_labels.cpu().numpy())

In [ ]:
# Calculate Accuracy

bert_accuracy = accuracy_score(true_labels, predictions)

print("BERT Test Accuracy:", bert_accuracy)

In [ ]:
# Classification Report

print(classification_report(true_labels, predictions))

In [ ]:
# Confusion Matrix

bert_cm = confusion_matrix(true_labels, predictions)

print(bert_cm)